## Disk space — `df` and `du`

Two different questions, two different tools.

**`df` — "how full are my filesystems?"** Shows usage **per mounted filesystem**:

```bash
df -h        # human-readable sizes
df -hT       # ...and the filesystem type
df -i        # inode usage instead of bytes
```

Columns: **Size**, **Used**, **Avail** (what's left for *non-root* users), **Use%**, **Mounted on**. Two surprises to remember:

1. **The 5% root reserve** — on ext4, 5% is reserved for root, so `df` can read "95% used" while regular users already can't write. Drop it with `tune2fs -m 1 /dev/sda1`.
2. **Inodes can run out before bytes.** Each file uses one inode, fixed at format time; millions of tiny files can exhaust inodes with bytes to spare. `df -i` reveals it — the giveaway is "No space left on device" *with* free space.

**`du` — "what's using space inside this directory?"** Walks a tree and totals each item:

```bash
du -sh /var/log                 # one summary for the whole tree
du -h --max-depth=1 /var/log    # one level deep — find big subdirs
du -sh /var/log/* | sort -h     # sorted by size — the useful form
```

The "find what's eating my disk" combo: `du -sh /var/log/* 2>/dev/null | sort -h | tail`.

⚠️ **`df` says full but `du` disagrees?** The classic cause is a **deleted-but-still-open file**: a process holds it open, so `rm` frees no space until the process closes it. Find them with `lsof | grep deleted`. (Modern helper: **`ncdu`**, an interactive size browser.)
